# Capstone Project 3
# Agent Governance, Observability, ERP Posting & Executive Reporting
### "Deliver & Govern" layer on top of Capstones 1 & 2

**Maps to curriculum tracks:** Agent Maturity & Delivery Lifecycle, Risk in Agentic Systems,
Responsible AI & Governance, Operating Agents at Scale, Rollout/RAID & Delivery Tabletop,
Portfolio Governance & Reuse — plus the Day 3 use cases *Partial Payments*, *ERP Integration
(D365)*, and *Reporting*.

---

## 1. Problem Statement

Capstone 1 built an **ingestion agent system** and Capstone 2 built a **payment-matching agent
system**. Before either goes to production, an enterprise governance function needs to:

1. **Score agent maturity** against a repeatable rubric (not "does it work on my laptop", but
   data quality, HITL coverage, observability, test coverage, rollback readiness).
2. Maintain a **RAID/risk register** for agentic systems specifically — model risk, data risk,
   operational risk, compliance risk — with a heat-matrix view for the governance board.
3. Run **Responsible AI guardrail checks** on live agent output: PII leakage scanning,
   confidence-threshold audits, and a basic fairness check (auto-approval rate should not vary
   wildly by customer segment without explanation).
4. Take matched payments from Capstone 2 and generate valid **ERP journal entries** (D365-style:
   balanced debit/credit lines) ready for posting, plus a mocked posting call.
5. Assemble everything into a **reporting agent** that produces a stakeholder-ready executive
   summary — volumes, match rates, exceptions, risk heat matrix, governance flags — in one file.

**Goal:** operationalise the "Deliver & Govern" module as code, not just a slide deck, so a
portfolio of agents (Capstone 1, Capstone 2, and future ones) can be governed consistently and
reported on automatically.


In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────
import re
import json
from dataclasses import dataclass
from datetime import datetime
from typing import List, Dict, Any

import pandas as pd
import numpy as np

pd.set_option("display.max_colwidth", 100)


## 2. Inputs Carried Over From Capstones 1 & 2

In a real pipeline these would be read from a data store / message bus. Here we recreate the
final outputs of the two prior notebooks directly so this notebook is self-contained and
runnable on its own.


In [ ]:
# Output of Capstone 1 (remittance ingestion) — one row per document processed
ingestion_results = pd.DataFrame([
    {"record_id": "a1b2c3d4", "source_type": "email",   "customer": "Acme Corp",         "amount": 12450.00, "confidence": 0.95, "straight_through": True,  "segment": "Enterprise"},
    {"record_id": "b2c3d4e5", "source_type": "erp_csv",  "customer": "Contoso Retail Ltd", "amount": 8899.50,  "confidence": 0.98, "straight_through": True,  "segment": "Mid-Market"},
    {"record_id": "c3d4e5f6", "source_type": "pdf",      "customer": "Fabrikam Inc",       "amount": 4950.25,  "confidence": 0.90, "straight_through": True,  "segment": "Mid-Market"},
    {"record_id": "d4e5f6g7", "source_type": "erp_csv",  "customer": "Northwind Traders",  "amount": None,     "confidence": 0.40, "straight_through": False, "segment": "SMB"},
    {"record_id": "e5f6g7h8", "source_type": "email",    "customer": "Unknown Vendor",     "amount": None,     "confidence": 0.15, "straight_through": False, "segment": "SMB"},
])

# Output of Capstone 2 (payment matching) — one row per payment resolved
matching_results = pd.DataFrame([
    {"payment_id": "PMT-01", "matched_invoices": ["INV-100234"],              "method": "exact_2way",       "confidence": 1.00, "variance": 0.00,   "amount": 12450.00, "segment": "Enterprise"},
    {"payment_id": "PMT-02", "matched_invoices": ["INV-100235"],              "method": "exact_2way",       "confidence": 1.00, "variance": 0.00,   "amount": 8899.50,  "segment": "Mid-Market"},
    {"payment_id": "PMT-03", "matched_invoices": ["INV-100236"],              "method": "fuzzy",            "confidence": 0.87, "variance": 0.00,   "amount": 4950.25,  "segment": "Mid-Market"},
    {"payment_id": "PMT-04", "matched_invoices": ["INV-100237", "INV-100238"], "method": "partial_allocation", "confidence": 0.85, "variance": -1200.00, "amount": 4000.00,  "segment": "SMB"},
    {"payment_id": "PMT-05", "matched_invoices": ["INV-100239"],              "method": "smart_ai_match",   "confidence": 0.41, "variance": 0.00,   "amount": 14950.00, "segment": "Enterprise"},
])

ingestion_results, matching_results


## 3. Case 1 — Agent Maturity Assessment

A simple, transparent rubric (0–5 per dimension) scored against evidence already produced by
Capstones 1 & 2, rather than a subjective "looks good to me" sign-off.


In [ ]:
MATURITY_DIMENSIONS = [
    "data_quality_controls", "hitl_coverage", "observability",
    "automated_test_coverage", "rollback_readiness",
]

def score_maturity(agent_name: str, evidence: Dict[str, int]) -> Dict[str, Any]:
    scores = {dim: evidence.get(dim, 0) for dim in MATURITY_DIMENSIONS}
    overall = round(sum(scores.values()) / len(scores), 2)
    level = (
        "5 - Optimized" if overall >= 4.5 else
        "4 - Managed" if overall >= 3.5 else
        "3 - Defined" if overall >= 2.5 else
        "2 - Repeatable" if overall >= 1.5 else
        "1 - Ad hoc"
    )
    return {"agent": agent_name, **scores, "overall_score": overall, "maturity_level": level}

# Evidence-based scoring, derived from what Capstones 1 & 2 actually demonstrated:
maturity_rows = [
    score_maturity("Capstone1_RemittanceIngestion", {
        "data_quality_controls": 4,   # validation_node rejects incomplete records
        "hitl_coverage": 5,           # conditional edge routes every low-confidence case to HITL
        "observability": 3,           # has a trace log, no metrics dashboard yet
        "automated_test_coverage": 3, # 5 scripted cases, no CI regression suite
        "rollback_readiness": 2,      # no versioned tool contracts / rollback plan yet
    }),
    score_maturity("Capstone2_PaymentMatching", {
        "data_quality_controls": 4,
        "hitl_coverage": 4,           # smart-match routes low-confidence to HITL, others don't explicitly
        "observability": 5,           # structured decision log + per-agent dashboard
        "automated_test_coverage": 5, # golden-set precision/recall/F1 harness
        "rollback_readiness": 3,
    }),
]

maturity_df = pd.DataFrame(maturity_rows)
maturity_df


## 4. Case 2 — RAID / Risk Register Automation

Generates risk-register entries classified by category (model / data / operational /
compliance), with a likelihood × severity heat score for the governance board.


In [ ]:
@dataclass
class RiskEntry:
    risk_id: str
    agent: str
    category: str          # model | data | operational | compliance
    description: str
    likelihood: int        # 1-5
    severity: int          # 1-5
    mitigation: str

    @property
    def heat_score(self) -> int:
        return self.likelihood * self.severity


risk_register = [
    RiskEntry("RSK-001", "Capstone1_RemittanceIngestion", "data",
              "Malformed/incomplete ERP rows silently mis-extracted", 3, 3,
              "Validation agent + mandatory-field schema check before finalize"),
    RiskEntry("RSK-002", "Capstone1_RemittanceIngestion", "model",
              "Regex-based extractor misses non-standard invoice formats", 4, 2,
              "Track extraction confidence; expand regex/NER coverage from HITL corrections"),
    RiskEntry("RSK-003", "Capstone2_PaymentMatching", "model",
              "Smart-match model trained on small synthetic set may not generalize", 3, 4,
              "Retrain periodically on labeled production HITL decisions; monitor drift"),
    RiskEntry("RSK-004", "Capstone2_PaymentMatching", "operational",
              "Partial-payment short-pay variance not routed to collections automatically", 3, 3,
              "Add downstream workflow trigger when variance exceeds materiality threshold"),
    RiskEntry("RSK-005", "Capstone2_PaymentMatching", "compliance",
              "Auto-approved matches lack a documented approval trail for audit", 2, 4,
              "Persist observability log to immutable store; retain per SOX record-retention policy"),
]

risk_df = pd.DataFrame([r.__dict__ | {"heat_score": r.heat_score} for r in risk_register])
risk_df.sort_values("heat_score", ascending=False)


In [ ]:
# Heat matrix: likelihood (rows) x severity (cols) -> count of risks in each cell
heat_matrix = risk_df.pivot_table(index="likelihood", columns="severity",
                                   values="risk_id", aggfunc="count", fill_value=0)
heat_matrix = heat_matrix.reindex(index=[5, 4, 3, 2, 1], columns=[1, 2, 3, 4, 5], fill_value=0)
heat_matrix


## 5. Case 3 — Responsible AI Governance Checks

Three automated guardrail checks run against the Capstone 1/2 outputs:

1. **PII leakage scan** — flags anything that looks like an email address or card-like number
   leaking into a field that shouldn't hold it.
2. **Confidence-threshold audit** — flags any `straight_through=True` record whose confidence is
   below the policy floor (should never happen if Capstone 1's routing logic is correct — this is
   exactly the kind of regression check governance would run continuously).
3. **Segment fairness check** — compares auto-approval / straight-through rate by customer
   segment; large unexplained gaps get flagged for review rather than silently accepted.


In [ ]:
PII_EMAIL_RE = re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+")
CARD_LIKE_RE = re.compile(r"\b(?:\d[ -]?){13,19}\b")
CONFIDENCE_FLOOR = 0.6

governance_flags = []

# Check 1: PII leakage in the 'customer' field (should hold a name, not an email/card number)
for _, row in ingestion_results.iterrows():
    if PII_EMAIL_RE.search(str(row["customer"])) or CARD_LIKE_RE.search(str(row["customer"])):
        governance_flags.append({"check": "pii_leakage", "record_id": row["record_id"],
                                  "detail": f"Possible PII in customer field: {row['customer']}"})

# Check 2: straight-through records must meet the confidence floor
bad_st = ingestion_results[(ingestion_results["straight_through"]) & (ingestion_results["confidence"] < CONFIDENCE_FLOOR)]
for _, row in bad_st.iterrows():
    governance_flags.append({"check": "confidence_floor_violation", "record_id": row["record_id"],
                              "detail": f"straight_through=True but confidence={row['confidence']}"})

# Check 3: segment fairness on Capstone 2 auto-approval (confidence >= 0.8) rate
matching_results["auto_approved"] = matching_results["confidence"] >= 0.8
seg_rates = matching_results.groupby("segment")["auto_approved"].mean()
spread = seg_rates.max() - seg_rates.min()
if spread > 0.4:
    governance_flags.append({"check": "segment_fairness", "record_id": "ALL",
                              "detail": f"Auto-approval rate spread across segments = {spread:.0%}: {seg_rates.to_dict()}"})

governance_df = pd.DataFrame(governance_flags)
print("Segment auto-approval rates:\n", seg_rates.round(2), "\n")
governance_df


## 6. Case 4 — ERP Integration: Journal Entry Generation & Posting (D365-style)

Converts matched payments into balanced double-entry journal lines
(`Dr Cash / Cr Accounts Receivable`), validates debit = credit per payment, and simulates a
posting call to a D365-style API.


In [ ]:
def build_journal_lines(matching_results: pd.DataFrame) -> pd.DataFrame:
    lines = []
    for _, pay in matching_results.iterrows():
        # Skip unmatched / zero-confidence rows — governance requires a match before posting
        if not pay["matched_invoices"] or pay["confidence"] == 0:
            continue
        je_id = f"JE-{pay['payment_id']}"
        amount = round(pay["amount"], 2)
        lines.append({"journal_id": je_id, "line": 1, "account": "1000-CASH",
                       "description": f"Cash receipt {pay['payment_id']}",
                       "debit": amount, "credit": 0.00})
        # If there's a variance (short-pay), the credit side splits between AR clearance and a
        # variance/write-off suspense account so debits still equal credits.
        ar_credit = amount + max(0, -pay["variance"])  # add back any short-pay to fully clear AR
        lines.append({"journal_id": je_id, "line": 2, "account": "1200-AR",
                       "description": f"AR clearance for {', '.join(pay['matched_invoices'])}",
                       "debit": 0.00, "credit": round(ar_credit, 2)})
        if pay["variance"] != 0:
            lines.append({"journal_id": je_id, "line": 3, "account": "6900-VARIANCE-SUSPENSE",
                           "description": f"Short-pay variance for {pay['payment_id']}",
                           "debit": max(0, -pay["variance"]), "credit": max(0, pay["variance"])})
    return pd.DataFrame(lines)


journal_df = build_journal_lines(matching_results)
journal_df


In [ ]:
# Validate every journal balances (sum debit == sum credit per journal_id)
balance_check = journal_df.groupby("journal_id")[["debit", "credit"]].sum()
balance_check["balanced"] = np.isclose(balance_check["debit"], balance_check["credit"], atol=0.01)
assert balance_check["balanced"].all(), "Unbalanced journal entry detected -- do not post!"
balance_check


In [ ]:
def mock_post_to_d365(journal_df: pd.DataFrame) -> List[dict]:
    """Simulated D365 Finance journal-import API call. In production this would call the
    real OData/Journal-import endpoint (or an MCP tool wrapping it, per A3) with retries and
    idempotency keys; here we just validate the payload shape and return mock posting receipts."""
    receipts = []
    for je_id, grp in journal_df.groupby("journal_id"):
        payload = grp[["account", "description", "debit", "credit"]].to_dict("records")
        receipts.append({"journal_id": je_id, "status": "POSTED", "lines_posted": len(payload),
                          "posted_at": datetime.now().isoformat()})
    return receipts


posting_receipts = mock_post_to_d365(journal_df)
pd.DataFrame(posting_receipts)


## 7. Case 5 — Automated Reporting Agent

Assembles every metric produced above into one stakeholder-ready executive summary. The
"agent" here is simply a function that pulls from each governed data source and writes a single
markdown report — the same aggregation pattern that would sit behind a scheduled reporting job.


In [ ]:
def generate_executive_report() -> str:
    st_rate = ingestion_results["straight_through"].mean()
    match_rate = (matching_results["confidence"] >= 0.6).mean()
    total_variance = matching_results["variance"].sum()
    top_risks = risk_df.sort_values("heat_score", ascending=False).head(3)

    report_lines = [
        f"# Executive Summary -- Agentic Remittance & Payment Matching Program",
        f"_Generated: {datetime.now().strftime('%Y-%m-%d %H:%M UTC')}_",
        "",
        "## 1. Volume & Straight-Through Processing",
        f"- Documents ingested: {len(ingestion_results)}",
        f"- Straight-through rate: {st_rate:.0%}",
        f"- Payments resolved: {len(matching_results)}",
        f"- Confident-match rate (>=0.6 conf): {match_rate:.0%}",
        f"- Net matching variance (short-pay/over-pay): {total_variance:,.2f}",
        "",
        "## 2. Agent Maturity",
        maturity_df[["agent", "overall_score", "maturity_level"]].to_markdown(index=False),
        "",
        "## 3. Top Risks (by heat score)",
        top_risks[["risk_id", "agent", "category", "description", "heat_score"]].to_markdown(index=False),
        "",
        "## 4. Responsible AI Governance Flags",
        (governance_df.to_markdown(index=False) if not governance_df.empty else "_No governance flags raised this cycle._"),
        "",
        "## 5. ERP Posting Status",
        pd.DataFrame(posting_receipts).to_markdown(index=False),
        "",
        "## 6. Recommendation",
        "- Promote Capstone 2 (Payment Matching) toward pilot: highest maturity score, full "
        "observability and test coverage.",
        "- Hold Capstone 1 (Remittance Ingestion) at current stage until rollback readiness and "
        "observability dashboards are improved (see RSK-002).",
        "- Route RSK-003 and RSK-005 to the governance board given heat scores >= 8.",
    ]
    return "\n".join(str(x) for x in report_lines)


report_text = generate_executive_report()
print(report_text[:1500], "...\n[truncated for display]")


In [ ]:
import os
os.makedirs("/mnt/user-data/outputs", exist_ok=True)
with open("/mnt/user-data/outputs/executive_summary_report.md", "w") as f:
    f.write(report_text)
journal_df.to_csv("/mnt/user-data/outputs/journal_entries.csv", index=False)
risk_df.to_csv("/mnt/user-data/outputs/risk_register.csv", index=False)
print("Report and supporting CSVs written to /mnt/user-data/outputs/")


## 8. End Result & Conclusion

- The **maturity scoring** shows Capstone 2 (Payment Matching) is closer to production-ready
  (higher observability + test coverage) than Capstone 1 (Ingestion), giving portfolio governance
  a defensible, evidence-based sequencing decision rather than a gut call.
- The **risk register + heat matrix** turns "agentic systems are risky" into five specific,
  owned, mitigated risks a governance board can actually act on.
- The **Responsible AI checks** ran automatically against real pipeline output and would catch
  regressions (a PII leak, a confidence-floor breach, or a fairness gap) before they reach
  production — this is the guardrail layer that makes autonomous straight-through processing
  defensible.
- The **ERP journal entries balance by construction** (assert-checked) before the mocked D365
  posting call, which is the non-negotiable control for any finance-adjacent agent.
- The **reporting agent** produces one artifact a controller, program sponsor, or governance
  board can read in five minutes, drawing on every other agent's output — this is what "operating
  agents at scale" looks like in practice: one governed pipeline, one report, many contributing
  agents.

## 9. Applications

- **Agent Center of Excellence / portfolio governance** — apply the same maturity rubric and risk
  register template to every agent a bank, insurer, or enterprise IT function ships.
- **Internal audit & SOX controls** — the observability log + balanced journal entries + approval
  trail directly support audit evidence requirements for automated financial postings.
- **Finance transformation programs (D365 / SAP S/4HANA)** — the journal-generation pattern
  generalizes to any sub-ledger-to-GL posting problem (AP, fixed assets, intercompany).
- **Model risk management** — the RAID register and confidence-floor checks map directly onto
  standard MRM frameworks (e.g. SR 11-7-style governance) applied to LLM/agentic systems.
- **Executive/board reporting** — the reporting-agent pattern (pull from N governed sources,
  render one document) generalizes to any multi-agent program status report.
